# Deep Reinforcement Learning, Hands-On
## From the Agent–Environment Loop to Advantage Actor-Critic (CartPole)

> *"Machine learning ≈ looking for a function."* — Hung-yi Lee, Deep RL lecture

In reinforcement learning (RL) that function is a **policy**: an **actor** that maps an **observation** to an **action** while interacting with an **environment** that returns a **reward** — a reward that is often *delayed*. The goal is to find the policy that **maximizes the expected total reward**.

This notebook walks the lecture's spine, end to end, on one tiny environment (**CartPole-v1**) so every idea runs in a class session on a free Colab **CPU**:

| Section | Idea | Objective |
|---------|------|-----------|
| 1 | What is RL? — action = f(observation) | **LO1** |
| 2 | The actor is a classifier over actions | **LO2** |
| 3 | Return & reward delay | **LO3** |
| 4 | Policy Gradient / REINFORCE (V0→V3) | **LO4** |
| 5 | The critic: Monte-Carlo vs Temporal-Difference | **LO5** |
| 6 | Advantage Actor-Critic (A2C, V3.5→V4) | **LO6** |

**Demo philosophy:** one small environment, every concept executable, each interactive widget isolating exactly one pedagogical variable.

## Part 0 — Setup

The demo uses **Gymnasium's CartPole-v1** with **PyTorch autograd** for the policy and value networks — matching the lecture's gradient-descent / cross-entropy framing. Everything runs on a **free Colab CPU** in seconds to minutes; **no GPU is needed**.

The only install is `gymnasium` and `ipywidgets` (torch / numpy / matplotlib are already on Colab). Random seeds are fixed via `set_seed()` so figures and comparisons are reproducible across reruns.

In [ ]:
# Part 0 — Setup. Only gymnasium and ipywidgets need installing;
# torch / numpy / matplotlib are already preinstalled on Colab.
!pip install -q gymnasium ipywidgets

import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
import gymnasium as gym
import ipywidgets as widgets
from torch.distributions import Categorical
from IPython.display import display, clear_output

# Inline plotting (guarded so the cell also runs outside a notebook).
try:
    get_ipython().run_line_magic('matplotlib', 'inline')
except Exception:
    pass

# CartPole is tiny: CPU is correct and avoids device-mismatch bugs.
device = torch.device('cpu')

SEED = 0

def set_seed(seed: int = 0) -> None:
    """Seed Python, NumPy and PyTorch for reproducible runs."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

set_seed(SEED)

# Readable matplotlib defaults.
plt.rcParams.update({
    'figure.figsize': (7, 4),
    'axes.titlesize': 13,
    'axes.labelsize': 11,
    'legend.fontsize': 10,
    'figure.dpi': 110,
})

print(f"torch {torch.__version__} | device={device} | gymnasium {gym.__version__}")

## 1 · Reinforcement learning is "looking for a function"

In supervised learning we have per-step labels. In RL we **do not** — the only feedback is a scalar **reward**, and it can be **delayed**. We still "look for a function":

$$\text{Action} = f(\text{Observation})$$

The actor observes a state, picks an action, the environment responds with the next state and a reward, and the loop repeats. The objective is to choose $f$ so the **expected total reward** over an episode is as large as possible.

**Running environment — CartPole-v1:**

- **Observation:** 4 floats — cart position, cart velocity, pole angle, pole angular velocity.
- **Actions:** 2 discrete — push **left (0)** or **right (1)**.
- **Reward:** **+1 for every step the pole stays up** (episode caps at 500).

There are no labels telling us the "correct" action — only the reward stream. That is the whole challenge.

In [ ]:
# Instantiate the environment used for the whole notebook.
env = gym.make('CartPole-v1')

print('observation_space:', env.observation_space)
print('  -> 4 floats: cart pos/vel, pole angle/ang-vel')
print('action_space:    ', env.action_space)
print('  -> 2 actions: 0 = push left, 1 = push right; +1 reward per surviving step')

def run_random_episode(env, max_steps: int = 500, n_print: int = 5, verbose: bool = True) -> float:
    """Run one CartPole episode with uniformly random actions; return the total return."""
    obs, info = env.reset(seed=SEED)
    total_reward = 0.0
    for step in range(max_steps):
        action = env.action_space.sample()
        next_obs, reward, terminated, truncated, info = env.step(action)
        total_reward += reward
        if verbose and step < n_print:
            print(f'  t={step}: state={np.round(obs, 3)}, action={action}, reward={reward}')
        obs = next_obs
        if terminated or truncated:
            break
    return float(total_reward)

print('\nOne annotated random rollout:')
ret = run_random_episode(env, verbose=True)
print(f'total return = {ret}')

In [ ]:
DEFAULT_EPISODES = 50

def plot_random_returns(number_of_episodes: int = 50) -> None:
    """Histogram of returns from `number_of_episodes` random-policy episodes."""
    number_of_episodes = max(1, int(number_of_episodes))
    returns = [run_random_episode(env, verbose=False) for _ in range(number_of_episodes)]
    mean_ret = float(np.mean(returns))
    plt.figure()
    bins = min(30, max(5, number_of_episodes // 5))
    plt.hist(returns, bins=bins, color='#4C72B0', edgecolor='white', alpha=0.85)
    plt.axvline(mean_ret, color='crimson', linestyle='--', linewidth=2,
                label=f'mean = {mean_ret:.1f}')
    plt.axvline(500, color='gray', linestyle=':', linewidth=1.5, label='cap = 500')
    plt.title(f'Random-policy returns (mean={mean_ret:.1f})')
    plt.xlabel('episode return')
    plt.ylabel('count')
    plt.legend()
    plt.show()

# Static default so a figure always renders, even if widgets fail to initialize.
plot_random_returns(DEFAULT_EPISODES)

# Interactive: slide the number of episodes to redraw the histogram.
try:
    widgets.interact(
        plot_random_returns,
        number_of_episodes=widgets.IntSlider(min=10, max=500, step=10, value=DEFAULT_EPISODES),
    )
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## 2 · The actor is a classifier  *(RL Step 1: a function with unknowns)*

The policy network is just a **classifier over actions**:

- **input:** the 4-D observation,
- **output:** one neuron per action → a **softmax** giving a probability for each action,
- **action:** **sampled** from that distribution — *not* `argmax`.

Why sample instead of taking the most probable action? Because a **stochastic policy explores**. Early on the network knows nothing; sampling lets it try both actions and discover which leads to more reward. As training proceeds the distribution sharpens toward good actions on its own.

In [ ]:
class PolicyNet(nn.Module):
    """Actor: a small MLP mapping a 4-D observation to action logits."""
    def __init__(self, obs_dim: int = 4, hidden: int = 128, n_actions: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, n_actions),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # Return raw logits; softmax is applied at sampling time for numerical stability.
        return self.net(x)

def select_action(policy, state):
    """Sample an action from the policy's softmax. Returns (action:int, log_prob:tensor)."""
    state = torch.as_tensor(state, dtype=torch.float32, device=device)
    if state.dim() == 1:
        state = state.unsqueeze(0)
    logits = policy(state)
    dist = Categorical(logits=logits)
    action = dist.sample()
    # Keep log_prob on the autograd graph for the policy-gradient loss.
    return int(action.item()), dist.log_prob(action).squeeze()

# Default (untrained) actor, reused by the visualizations and training cells below.
set_seed(SEED)
policy = PolicyNet().to(device)
print(policy)

In [ ]:
# One fixed observation to probe the (untrained) actor.
sample_state, _ = env.reset(seed=SEED)

def current_probs() -> np.ndarray:
    """Current softmax action probabilities for the fixed sample_state."""
    s = torch.as_tensor(sample_state, dtype=torch.float32, device=device).unsqueeze(0)
    probs = torch.softmax(policy(s), dim=-1)
    return probs.detach().cpu().numpy().ravel()

def draw_action_probs(highlight_action=None) -> None:
    """Bar chart of action probabilities; highlight the sampled action if given."""
    probs = current_probs()
    labels = ['left (0)', 'right (1)']
    colors = ['#4C72B0', '#4C72B0']
    if highlight_action is not None:
        colors[highlight_action] = '#DD8452'
    plt.figure()
    bars = plt.bar(labels, probs, color=colors, edgecolor='white')
    for b, p in zip(bars, probs):
        plt.text(b.get_x() + b.get_width() / 2, p + 0.02, f'{p:.2f}', ha='center', va='bottom')
    plt.ylim(0, 1)
    plt.ylabel('probability')
    title = 'Actor = classifier over actions'
    if highlight_action is not None:
        title += f'  ->  sampled action: {labels[highlight_action]}'
    plt.title(title)
    plt.show()

def on_sample_clicked(b) -> None:
    with sample_out:
        clear_output(wait=True)
        a, _ = select_action(policy, sample_state)
        draw_action_probs(highlight_action=a)

# Static default render (no highlight) so a chart shows on load.
draw_action_probs()

# Button resamples an action from the current softmax and highlights it.
try:
    sample_button = widgets.Button(description='sample action')
    sample_out = widgets.Output()
    sample_button.on_click(on_sample_clicked)
    display(sample_button, sample_out)
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## 3 · What are we maximizing? Return and reward delay  *(RL Step 2: the loss)*

A full run is a **trajectory** $\tau = (s_1, a_1, s_2, a_2, \dots)$. Its **return** is the sum of rewards collected along the way:

$$R(\tau) = \sum_t r_t$$

We want to **maximize** $R$, so the "loss" we minimize is essentially the **negative return**.

But this optimization is *not* standard supervised learning:

- the environment and the sampled actions form a **black box with randomness**, and
- rewards are **delayed** — a good move now may only pay off many steps later. (The lecture's short-sighted "always fire" example sacrifices a larger long-term reward for an immediate one.)

So we cannot simply backprop a per-step label; we need the **policy gradient**.

In [ ]:
def rollout_episode(policy, env, max_steps: int = 500) -> dict:
    """Run one episode with the policy; collect states/actions/rewards/log_probs and the return."""
    obs, _ = env.reset()
    states, actions, rewards, log_probs = [], [], [], []
    for _ in range(max_steps):
        action, log_prob = select_action(policy, obs)
        next_obs, reward, terminated, truncated, _ = env.step(action)
        states.append(np.asarray(obs, dtype=np.float32).copy())
        actions.append(action)
        rewards.append(float(reward))
        log_probs.append(log_prob)
        obs = next_obs
        if terminated or truncated:
            break
    return {
        'states': states,
        'actions': actions,
        'rewards': rewards,
        'log_probs': log_probs,
        'total_return': float(sum(rewards)),
    }

set_seed(SEED)
trajectory = rollout_episode(policy, env)
print(f"trajectory length = {len(trajectory['rewards'])} steps, "
      f"total return = {trajectory['total_return']}")

In [ ]:
def discounted_reward_to_go(rewards, gamma: float) -> np.ndarray:
    """G'_t = r_t + gamma * G'_{t+1}, computed backward over the reward sequence."""
    r = np.asarray(rewards, dtype=float)
    if len(r) == 0:
        return r
    rtg = np.zeros_like(r)
    running = 0.0
    for t in range(len(r) - 1, -1, -1):
        running = r[t] + gamma * running
        rtg[t] = running
    return rtg

def plot_reward_to_go(gamma: float = 0.99) -> None:
    """Plot per-step immediate reward vs discounted reward-to-go for one trajectory."""
    gamma = float(min(1.0, max(0.0, gamma)))
    rewards = trajectory['rewards']
    rtg = discounted_reward_to_go(rewards, gamma)
    steps = np.arange(len(rewards))
    plt.figure()
    plt.step(steps, rewards, where='mid', color='gray', label='immediate reward $r_t$')
    plt.plot(steps, rtg, color='#C44E52', marker='o', markersize=3, label="reward-to-go $G'_t$")
    plt.title(f'Reward delay: gamma={gamma:.2f}')
    plt.xlabel('time step t')
    plt.ylabel('reward / reward-to-go')
    plt.legend()
    plt.show()

# Static default.
plot_reward_to_go(0.99)

# Slide gamma to see how far credit propagates back in time.
try:
    widgets.interact(
        plot_reward_to_go,
        gamma=widgets.FloatSlider(min=0.0, max=1.0, step=0.01, value=0.99),
    )
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## 4 · Policy Gradient / REINFORCE

We shape a **weighted cross-entropy** loss — encourage the actions that led to good outcomes, discourage the rest:

$$L = \sum_n A_n \, e_n, \qquad e_n = -\log \pi(a_n \mid s_n)$$

Everything hinges on the **advantage weight** $A_n$. The lecture builds it up in versions:

| Version | $A_n$ | Idea |
|---------|-------|------|
| **V0** | $r_n$ | immediate reward — short-sighted |
| **V1** | $G_t = \sum_{n\ge t} r_n$ | cumulative return |
| **V2** | $G'_t = \sum_{n\ge t}\gamma^{\,n-t} r_n$ | **discounted** return |
| **V3** | $G'_t - b$ | subtract a **baseline** so advantages become signed |

REINFORCE is **on-policy**: each update needs **fresh** rollouts from the *current* policy — we cannot reuse old data.

In [ ]:
def compute_advantage(rewards, version: str = 'V2', gamma: float = 0.99,
                      use_baseline: bool = False) -> np.ndarray:
    """Per-step advantage weight A_n for the four REINFORCE versions.

    V0: A_n = r_n                 (immediate reward, short-sighted)
    V1: A_n = G_t                 (undiscounted cumulative reward-to-go)
    V2: A_n = G'_t                (discounted reward-to-go)
    V3: A_n = G'_t - baseline     (centered & normalized so advantages are signed)
    """
    if version not in {'V0', 'V1', 'V2', 'V3'}:
        raise ValueError(f'version must be one of V0..V3, got {version!r}')
    r = np.asarray(rewards, dtype=float)
    if len(r) == 0:
        return r
    if version == 'V0':
        A = r.copy()
    elif version == 'V1':
        A = discounted_reward_to_go(r, 1.0)
    else:  # V2 and V3 both start from the discounted reward-to-go
        A = discounted_reward_to_go(r, gamma)
    if version == 'V3' or use_baseline:
        A = A - A.mean()                 # constant baseline b = mean(A)
        A = A / (A.std() + 1e-8)          # normalize for stable gradients
    return A

def policy_gradient_loss(log_probs, advantages) -> torch.Tensor:
    """Weighted cross-entropy loss L = -sum_n A_n * log pi(a_n | s_n)."""
    if len(log_probs) == 0:
        return torch.zeros((), device=device, requires_grad=True)
    lp = torch.stack(list(log_probs))
    adv = torch.as_tensor(np.asarray(advantages), dtype=torch.float32, device=device).detach()
    return -(lp * adv).sum()

# Show the four versions side by side on a fixed reward sequence.
sample_rewards = [1, 1, 1, 1, 1]
print(f'sample rewards: {sample_rewards}\n')
for v in ['V0', 'V1', 'V2', 'V3']:
    weights = np.round(compute_advantage(sample_rewards, version=v, gamma=0.99), 3)
    print(f'  {v}: {weights}')

In [ ]:
def train_policy_gradient(version: str = 'V2', gamma: float = 0.99,
                          use_baseline: bool = False, num_iters: int = 50,
                          lr: float = 1e-2, hidden: int = 128, seed: int = 0) -> list:
    """On-policy REINFORCE: fresh rollout + advantage + Adam update each iteration."""
    set_seed(seed)
    net = PolicyNet(hidden=hidden).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=lr)
    returns_curve = []
    for _ in range(num_iters):
        traj = rollout_episode(net, env)
        if len(traj['rewards']) == 0:
            returns_curve.append(0.0)
            continue
        adv = compute_advantage(traj['rewards'], version, gamma, use_baseline)
        loss = policy_gradient_loss(traj['log_probs'], adv)
        if not torch.isfinite(loss):
            print('[warning] non-finite loss, stopping early')
            break
        opt.zero_grad()
        loss.backward()
        opt.step()
        returns_curve.append(traj['total_return'])
    return returns_curve

# Short default run to confirm the return rises.
set_seed(SEED)
curve = train_policy_gradient(version='V2', num_iters=40)
plt.figure()
plt.plot(curve, color='#4C72B0')
plt.title('REINFORCE (V2) - default run')
plt.xlabel('iteration')
plt.ylabel('episode return')
plt.show()
print(f'first-3 mean = {np.mean(curve[:3]):.1f}, last-3 mean = {np.mean(curve[-3:]):.1f}')

In [ ]:
def moving_average(x, window: int) -> np.ndarray:
    """Simple moving average; window clamped to len(x)."""
    x = np.asarray(x, dtype=float)
    if len(x) == 0:
        return x
    window = max(1, min(int(window), len(x)))
    kernel = np.ones(window) / window
    return np.convolve(x, kernel, mode='same')

# Persistent shared figure so successive runs overlay for comparison.
fig, ax = plt.subplots()
ax.set_title('REINFORCE: advantage versions compared')
ax.set_xlabel('iteration')
ax.set_ylabel('episode return (smoothed)')
pg_runs = []

def run_pg_experiment(version: str = 'V2', gamma: float = 0.99,
                      use_baseline: bool = False, num_iters: int = 40) -> None:
    """Train a short REINFORCE and overlay its smoothed learning curve on the shared axes."""
    label = f'{version} g={gamma:.2f} b={use_baseline}'
    if label in pg_runs:
        return  # avoid duplicate curves / legend clutter
    curve = train_policy_gradient(version, gamma, use_baseline, num_iters=num_iters)
    smooth = moving_average(curve, window=min(5, len(curve)))
    ax.plot(smooth, label=label)
    ax.legend()
    pg_runs.append(label)
    display(fig)

def on_run_clicked(b) -> None:
    with pg_out:
        clear_output(wait=True)
        try:
            run_pg_experiment(version_dd.value, gamma_sl.value, baseline_cb.value, num_iters=40)
        except Exception as e:
            print(f'[run failed] {e}')

# Static default so a curve shows on load.
run_pg_experiment('V2', 0.99, False)

# Interactive controls: pick version / gamma / baseline, then Run.
try:
    version_dd = widgets.Dropdown(options=['V0', 'V1', 'V2', 'V3'], value='V2', description='version')
    gamma_sl = widgets.FloatSlider(min=0.5, max=1.0, step=0.01, value=0.99, description='gamma')
    baseline_cb = widgets.Checkbox(value=False, description='baseline')
    run_btn = widgets.Button(description='Run')
    pg_out = widgets.Output()
    run_btn.on_click(on_run_clicked)
    display(widgets.VBox([version_dd, gamma_sl, baseline_cb, run_btn, pg_out]))
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## 5 · The critic: estimating a value function  *(Monte-Carlo vs Temporal-Difference)*

A **critic** learns a **value function**

$$V_\theta(s) = \mathbb{E}\big[\text{discounted return after seeing } s \mid \text{actor } \theta\big].$$

Two ways to estimate it:

- **Monte-Carlo (MC):** regress $V(s_t)$ toward the **full-episode** return $G'_t$. Unbiased, but high variance (it waits for the episode to end).
- **Temporal-Difference (TD):** use the **bootstrap** $V(s_t) \approx r_t + \gamma\,V(s_{t+1})$. Lower variance, but biased by the current estimate.

The next cell reproduces **Sutton & Barto's Example 6.4** exactly: over 8 episodes, MC gives $V(s_a)=0$ while batch-TD gives $V(s_a)=\tfrac{3}{4}$ with $V(s_b)=\tfrac{3}{4}$ — the canonical demonstration that the two methods can disagree.

In [ ]:
# Canonical 8-episode dataset over states {'A','B'} (Sutton & Barto, Example 6.4).
# Each episode is a list of (state, reward) transitions ending in a terminal state.
sutton_episodes = [
    [('A', 0), ('B', 0)],   # episode 1: A -> B -> terminal, reward 0
    [('B', 1)],             # episodes 2-7: B -> terminal, reward 1 (six of them)
    [('B', 1)],
    [('B', 1)],
    [('B', 1)],
    [('B', 1)],
    [('B', 1)],
    [('B', 0)],             # episode 8: B -> terminal, reward 0
]

def mc_estimate(episodes, gamma: float = 1.0) -> dict:
    """Every-visit Monte-Carlo: average the (discounted) return-to-go per state."""
    sums, counts = {}, {}
    for ep in episodes:
        T = len(ep)
        for t in range(T):
            s, _ = ep[t]
            G = 0.0
            for k in range(t, T):
                G += (gamma ** (k - t)) * ep[k][1]
            sums[s] = sums.get(s, 0.0) + G
            counts[s] = counts.get(s, 0) + 1
    return {s: sums[s] / counts[s] for s in sums}

def td_estimate(episodes, gamma: float = 1.0, iters: int = 2000, alpha: float = 0.01) -> dict:
    """Batch / certainty-equivalence TD(0) fixed point via repeated sweeps over the data."""
    states = sorted({s for ep in episodes for (s, _) in ep})
    V = {s: 0.0 for s in states}
    for _ in range(iters):
        for ep in episodes:
            T = len(ep)
            for t in range(T):
                s, r = ep[t]
                v_next = V[ep[t + 1][0]] if t + 1 < T else 0.0  # terminal next-state value = 0
                V[s] += alpha * (r + gamma * v_next - V[s])
    return V

mc = mc_estimate(sutton_episodes)
td = td_estimate(sutton_episodes)
print('Sutton Example 6.4 - value estimates')
print(f"  Monte-Carlo:        V(A)={mc.get('A', 0.0):.3f}, V(B)={mc.get('B', 0.0):.3f}")
print(f"  Temporal-Diff (TD): V(A)={td.get('A', 0.0):.3f}, V(B)={td.get('B', 0.0):.3f}")
print('\nThey agree on B (0.75) but disagree on A: MC=0 vs TD=0.75.')

In [ ]:
def plot_mc_td(method: str = 'Temporal-Difference', episodes_observed: int = 8) -> None:
    """Bar chart of V(s_a), V(s_b) for the chosen estimator over the first k episodes."""
    episodes_observed = int(min(8, max(1, episodes_observed)))
    subset = sutton_episodes[:episodes_observed]
    if method == 'Monte-Carlo':
        vals = mc_estimate(subset)
    elif method == 'Temporal-Difference':
        vals = td_estimate(subset)
    else:  # unknown method -> default to TD and note it in the title
        method = 'Temporal-Difference'
        vals = td_estimate(subset)
    heights = [vals.get('A', 0.0), vals.get('B', 0.0)]
    plt.figure()
    bars = plt.bar(['V(s_a)', 'V(s_b)'], heights, color=['#DD8452', '#4C72B0'], edgecolor='white')
    for b, h in zip(bars, heights):
        plt.text(b.get_x() + b.get_width() / 2, h + 0.02, f'{h:.3f}', ha='center', va='bottom')
    plt.ylim(0, 1)
    plt.ylabel('value estimate')
    plt.title(f'{method}: first {episodes_observed} episodes')
    plt.show()

# Static default.
plot_mc_td('Temporal-Difference', 8)

# Toggle estimator and how many of the 8 episodes are observed.
try:
    widgets.interact(
        plot_mc_td,
        method=widgets.ToggleButtons(options=['Monte-Carlo', 'Temporal-Difference']),
        episodes_observed=widgets.IntSlider(min=1, max=8, step=1, value=8),
    )
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## 6 · Advantage Actor-Critic (A2C)

Now combine the two. The **critic becomes the baseline** for the actor:

- **V3.5:** $A_t = G'_t - V_\theta(s_t)$ — subtract the learned value from the MC return.
- **V4:** $A_t = r_t + \gamma\,V_\theta(s_{t+1}) - V_\theta(s_t)$ — the **TD advantage**.

Why does this help? Bootstrapping the critic **averages out the sampling noise** in the sampled return, so the advantage estimate has **much lower variance** than plain policy gradient — giving faster, steadier learning. In practice the actor and critic can even **share their lower layers**.

In [ ]:
class ValueNet(nn.Module):
    """Critic: a small MLP estimating the state value V(s)."""
    def __init__(self, obs_dim: int = 4, hidden: int = 128):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(obs_dim, hidden),
            nn.Tanh(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x).squeeze(-1)   # shape (batch,)

def train_a2c(advantage_mode: str = 'V4', gamma: float = 0.99, num_iters: int = 50,
              lr_actor: float = 1e-2, lr_critic: float = 1e-2, hidden: int = 128,
              seed: int = 0) -> list:
    """Advantage Actor-Critic.

    V3.5: A = G'_t - V(s_t)                      (Monte-Carlo baseline)
    V4:   A = r_t + gamma*V(s_{t+1}) - V(s_t)    (TD advantage)
    """
    if advantage_mode not in {'V3.5', 'V4'}:
        raise ValueError(f"advantage_mode must be 'V3.5' or 'V4', got {advantage_mode!r}")
    set_seed(seed)
    actor = PolicyNet(hidden=hidden).to(device)
    critic = ValueNet(hidden=hidden).to(device)
    opt_actor = torch.optim.Adam(actor.parameters(), lr=lr_actor)
    opt_critic = torch.optim.Adam(critic.parameters(), lr=lr_critic)
    returns_curve = []
    for _ in range(num_iters):
        traj = rollout_episode(actor, env)
        rewards = traj['rewards']
        if len(rewards) == 0:
            returns_curve.append(0.0)
            continue
        states = torch.as_tensor(np.stack(traj['states']), dtype=torch.float32, device=device)
        rewards_t = torch.as_tensor(rewards, dtype=torch.float32, device=device)
        values = critic(states)                       # (T,)
        log_probs = torch.stack(traj['log_probs'])    # (T,)
        if advantage_mode == 'V4':
            # Bootstrapped TD target: next-state value shifted by one, terminal = 0.
            next_values = torch.cat([values[1:], torch.zeros(1, device=device)])
            target = rewards_t + gamma * next_values.detach()
            advantage = (target - values).detach()
            critic_loss = F.mse_loss(values, target.detach())
        else:  # V3.5: Monte-Carlo return as both baseline and critic target
            G = torch.as_tensor(discounted_reward_to_go(rewards, gamma),
                                dtype=torch.float32, device=device)
            advantage = (G - values).detach()
            critic_loss = F.mse_loss(values, G.detach())
        actor_loss = -(log_probs * advantage).sum()
        opt_actor.zero_grad()
        opt_critic.zero_grad()
        (actor_loss + critic_loss).backward()
        opt_actor.step()
        opt_critic.step()
        returns_curve.append(traj['total_return'])
    return returns_curve

# Short default A2C run to confirm it learns.
set_seed(SEED)
a2c_curve = train_a2c('V4', num_iters=40)
plt.figure()
plt.plot(a2c_curve, color='#55A868')
plt.title('A2C (V4) - default run')
plt.xlabel('iteration')
plt.ylabel('episode return')
plt.show()
print(f'first-3 mean = {np.mean(a2c_curve[:3]):.1f}, last-3 mean = {np.mean(a2c_curve[-3:]):.1f}')

In [ ]:
def algo_runner(name: str, seed: int, num_iters: int) -> list:
    """Map an algorithm name to a training run with a given seed."""
    if name == 'REINFORCE V3':
        return train_policy_gradient(version='V3', use_baseline=True, num_iters=num_iters, seed=seed)
    if name == 'A2C V3.5':
        return train_a2c('V3.5', num_iters=num_iters, seed=seed)
    if name == 'A2C V4':
        return train_a2c('V4', num_iters=num_iters, seed=seed)
    raise ValueError(f'unknown algorithm {name!r}')

def run_comparison(algorithms=('REINFORCE V3', 'A2C V4'),
                   number_of_seeds: int = 3, num_iters: int = 40) -> None:
    """Overlay mean +/- std learning curves across seeds for each selected algorithm."""
    algorithms = list(algorithms)
    if not algorithms or number_of_seeds < 1:
        print('Select at least one algorithm and >= 1 seed.')
        return
    colors = {'REINFORCE V3': '#4C72B0', 'A2C V3.5': '#DD8452', 'A2C V4': '#55A868'}
    plt.figure()
    for name in algorithms:
        curves = [algo_runner(name, seed, num_iters) for seed in range(number_of_seeds)]
        L = min(len(c) for c in curves)
        arr = np.stack([c[:L] for c in curves])   # truncate to common length
        mean = arr.mean(axis=0)
        std = arr.std(axis=0)
        x = np.arange(L)
        col = colors.get(name)
        plt.plot(x, mean, label=name, color=col)
        plt.fill_between(x, mean - std, mean + std, alpha=0.2, color=col)
    plt.title('A2C reduces variance vs plain policy gradient')
    plt.xlabel('iteration')
    plt.ylabel('episode return')
    plt.legend()
    plt.show()

def on_compare_clicked(b) -> None:
    with cmp_out:
        clear_output(wait=True)
        try:
            run_comparison(list(algo_select.value), seeds_slider.value)
        except Exception as e:
            print(f'[comparison failed] {e}')

# Static default so an overlaid plot renders on load.
run_comparison(('REINFORCE V3', 'A2C V4'), 3)

# Choose algorithms and number of seeds, then Run.
try:
    algo_select = widgets.SelectMultiple(
        options=['REINFORCE V3', 'A2C V3.5', 'A2C V4'],
        value=('REINFORCE V3', 'A2C V4'),
        description='algos')
    seeds_slider = widgets.IntSlider(min=1, max=5, step=1, value=3, description='seeds')
    compare_btn = widgets.Button(description='Run')
    cmp_out = widgets.Output()
    compare_btn.on_click(on_compare_clicked)
    display(widgets.VBox([algo_select, seeds_slider, compare_btn, cmp_out]))
except Exception as e:
    print(f'[widget unavailable, showing static default] {e}')

## Recap & where to go next

We walked Hung-yi Lee's lecture spine, end to end, on one CartPole:

1. **RL = looking for a function** — action = f(observation), reward-driven, no labels. *(LO1)*
2. **Actor = classifier** — softmax over actions, sampled for exploration. *(LO2)*
3. **Return & reward delay** — maximize $R=\sum_t r_t$; $\gamma$ controls credit assignment. *(LO3)*
4. **Policy Gradient** — the advantage V0→V1→V2→V3; discounting and a baseline make it learn. *(LO4)*
5. **Critic, MC vs TD** — Sutton's Example 6.4: MC $V(s_a)=0$ vs TD $V(s_a)=3/4$. *(LO5)*
6. **Advantage Actor-Critic** — critic as baseline (V3.5/V4) cuts variance vs plain PG. *(LO6)*

**Intentionally left out** (to keep this notebook fast and executable) — worth exploring next:

- **On-policy vs off-policy & PPO** — reuse data safely with importance sampling + clipping.
- **Exploration tricks** — output-entropy regularization, parameter-space noise.
- **Reward shaping & curiosity** — intrinsic rewards when extrinsic reward is sparse.
- **Imitation learning / behavior cloning** and **inverse RL (IRL-as-GAN)**.
- **DQN** — the value-based branch of deep RL.

Each pairs naturally with the building blocks above — same actor-environment loop, richer objectives.